In [1]:
pip install chromadb ollama sentence-transformers -q

Note: you may need to restart the kernel to use updated packages.


In [2]:
import ollama
import chromadb
from chromadb.utils import embedding_functions

# === 1. Подготовка документов (ваши приватные данные) ===
# Это может быть: README, внутренняя wiki, Jira-описания, чек-листы.
# Важно: каждый элемент — независимый семантический фрагмент (чанк).
docs = [
    "FastAPI — это современный Python-фреймворк для создания API.",
    "Liquibase используется для управления миграциями базы данных.",
    "Docker упрощает развёртывание приложений через контейнеризацию."
]

# === 2. Настройка embedding-функции (ключевой компонент RAG) ===
# Используем специализированную модель для семантического поиска.
ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

# === 3. Сохранение в векторную БД Chroma (индексация) ===
client = chromadb.Client()

# Создаём коллекцию — аналог таблицы в реляционной БД.
# embedding_function автоматически преобразует документы в векторы при добавлении.
collection = client.create_collection(
    name="docs",
    embedding_function=ef  # Указываем, какую модель использовать для эмбеддингов
)

# Добавляем документы. Chroma:
# - вызовет ef.encode() для каждого документа,
# - сохранит вектор + исходный текст.
collection.add(
    ids=[f"id_{i}" for i in range(len(docs))],
    documents=docs
)


# === 4. Функция RAG-ответа (выполняется при каждом запросе) ===
def ask_rag(question: str) -> str:
    """
    Полный цикл RAG:
    1. Преобразовать вопрос в embedding (автоматически через ef),
    2. Найти самый релевантный фрагмент в Chroma,
    3. Передать его в LLM как контекст,
    4. Вернуть ответ от LLM.
    """
    # Шаг 1-2: семантический поиск
    # Chroma автоматически преобразует question в embedding той же моделью (ef),
    # затем ищет ближайший вектор и возвращает исходный текст.
    results = collection.query(
        query_texts=[question],
        n_results=1
    )
    context = results["documents"][0][0]

    # Шаг 3: генерация ответа через локальную LLM
    # Используем Ollama — локальный инференс без облака.
    # Mistral здесь работает ТОЛЬКО как генератор текста.
    # Она НЕ участвует в создании эмбеддингов!
    response = ollama.chat(
        model="mistral",
        messages=[
            {"role": "system", "content": "Отвечай только на основе следующего контекста. Если не знаешь ответа — скажи 'Не знаю'."},
            {"role": "user", "content": f"Контекст: {context}\n\nВопрос: {question}"}
        ],
        options={
            "temperature": 0.1
        }
    )
    return response.message.content


# === 4. Тест: запуск и проверка ===
if __name__ == "__main__":
    # Задаём вопрос, который явно ссылается на один из документов
    question = "Какой инструмент для миграций упомянут в документации?"
    answer = ask_rag(question)
    print(f"Вопрос: {question}")
    print(f"Ответ: {answer}")

/opt/anaconda3/envs/LLM_course/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5791.37it/s]


Вопрос: Какой инструмент для миграций упомянут в документации?
Ответ:  Liquibase


**Важно:** в этом примере embedding-модель и LLM — разные компоненты:

* all-MiniLM-L6-v2 отвечает только за эмбеддинги (для документов и запроса),
* mistral отвечает только за генерацию текста.

Никогда не используйте разные модели для эмбеддингов документов и запроса — даже если обе «работают», их векторные пространства не совместимы, и семантический поиск даст случайные результаты.

И не используйте LLM (например, mistral) как embedding-модель — это технически возможно в Ollama (ollama.embed), но качество будет низким. Для эмбеддингов всегда выбирайте **специализированные модели:** all-MiniLM-L6-v2, bge-m3, Snowflake Arctic Embed и т.д.

#### Ключевые правила

1) **Embedding-модель для документов и запроса — одна и та же.**

Иначе векторы будут в «разных системах координат» — поиск не сработает.

2) **LLM не должна генерировать embeddings.**

Это медленно, дорого и неэффективно. Используйте специализированные модели (all-MiniLM-L6-v2).

3) **Всегда ограничивайте температуру (temperature=0.1) в продакшене.**

Креативность — враг точности.

4) **Всегда добавляйте в system-промпт:**
```   
«Отвечай только на основе контекста. Если не знаешь — скажи "Не знаю"».
Это снижает галлюцинации.
```

Что происходит на каждом этапе?

|ЭТАП | ВХОД | ВЫХОД | ТЕХНОЛОГИЯ|
| :--- | :--- | :--- | :--- |
|1. Индексация | Текст документации | Векторы + текст в Chroma|sentence-transformers + chromadb|
|2. Запрос|Строка от пользователя|Embedding запроса|sentence-transformers|
|3. Поиск|Embedding запроса|Релевантный фрагмент|chromadb.query()|
|4. Генерация|Контекст + запрос|Ответ LLM → передаётся пользователю|ollama.chat()|